# ROPG-KD Retriever Training

Stage 1 of Simurgh's two-stage training: knowledge-distillation fine-tuning of the Qwen3-Embedding-0.6B encoder with a LoRA adapter, distilling from the offline LLM-judge teacher scores in `data/ropg_kd/{train,val}.jsonl` (see `notebooks/gen_ropg_data.ipynb`); checkpoints are selected each epoch on validation Recall@K / MRR per persona, not on training or validation KD loss.

**Kaggle setup checklist**
1. Enable GPU accelerator (T4 x1 is enough; ~10-20 min for 5 epochs).
2. Enable internet access (the encoder is downloaded from Hugging Face).
3. Attach the `simurgh-data` dataset, **version 2 or later** (must contain `ropg_kd/`).
4. No API secrets needed — training makes no LLM calls.

**Colab setup checklist**
1. Set `RUNTIME = "colab"` in the Config cell below.
2. Upload `simurgh-data/` to Google Drive at `MyDrive/simurgh-data/` — must contain `ropg_kd/train.jsonl`, `ropg_kd/val.jsonl`, `chunks/corpus.jsonl`.
3. Enable GPU accelerator (T4 × 1 is enough; ~10–20 min for 5 epochs).
4. No API secrets needed.
5. Checkpoints are saved directly to Google Drive — the final zip cell is skipped automatically.

In [ ]:
# Kaggle preinstalls torchao 0.10, which peft's LoRA dispatcher rejects (needs >= 0.16).
# We don't use torchao layers, so remove it and peft will skip that dispatcher.
!pip uninstall -q -y torchao
!pip install -q "sentence-transformers==5.6.0" peft accelerate

## Config

In [ ]:
import os

# Must be set before torch initializes CUDA; mitigates fragmentation OOMs by
# letting the allocator grow segments instead of hunting for contiguous blocks.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from pathlib import Path

# ── Runtime selector ─────────────────────────────────────────────────────────
# Set RUNTIME to match where you are running this notebook.
RUNTIME = "kaggle"  # "kaggle" | "colab" | "local"
GDRIVE_BASE = "/content/drive/MyDrive/simurgh-data"  # Colab only

# ── Paths ─────────────────────────────────────────────────────────────────────
if RUNTIME == "kaggle":
    DATASET_SLUG = "simurgh-data"
    DATA_ROOT = f"/kaggle/input/datasets/alirezahsn/{DATASET_SLUG}"
    OUTPUT_DIR = "/kaggle/working/ropg_kd_checkpoints"
elif RUNTIME == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_ROOT = GDRIVE_BASE
    OUTPUT_DIR = f"{GDRIVE_BASE}/ropg_kd_checkpoints"
else:  # local
    DATA_ROOT = "data"
    OUTPUT_DIR = "data/ropg_kd_checkpoints"

# ── Inline config (mirrors configs/train_ropg.yaml) ─────────────────────────
CFG = {
    "data": {
        "train_path": f"{DATA_ROOT}/ropg_kd/train.jsonl",
        "val_path": f"{DATA_ROOT}/ropg_kd/val.jsonl",
        "corpus_path": f"{DATA_ROOT}/chunks/corpus.jsonl",
    },
    "embedder": {
        "model": "Qwen/Qwen3-Embedding-0.6B",
        "max_seq_length": 512,  # most chunks are <260 tokens; 8192 wastes O(n²) attention
        "attn_implementation": "sdpa",  # fused attention; works on T4 without flash-attn
    },
    "training": {
        "device": "cuda",  # cpu for local smoke runs
        "precision": "fp16",  # fp16 | bf16 | fp32 — fp16 for T4/P100, bf16 for A100+
        "gradient_checkpointing": False,  # set to True in config if OOM on T4
        "epochs": 5,
        "batch_size": 8,  # groups per optimizer step (gradient accumulation)
        "doc_micro_batch": 8,  # docs per forward within a group; caps peak activation memory
        "lr": 2.0e-4,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "grad_clip": 1.0,
        "kd_temperature": 1.0,
    },
    "lora": {
        "r": 8,
        "alpha": 16,
        "dropout": 0.1,
        "target_modules": ["q_proj", "v_proj"],
    },
    "eval": {
        "top_k": 5,  # matches inference retrieval.top_k
        "relevance_top_m": 3,  # relevant = top-m docs by teacher_score per group
    },
    "checkpoint_dir": OUTPUT_DIR,
    "seed": 42,
}

Path(CFG["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)

## Core classes / personas

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Profile:
    id: str
    split: str
    rendered: str


PERSONAS = {
    "crammer": Profile(
        id="crammer",
        split="train",
        rendered=(
            "A ninth-grader who finds the textbook hard to follow and has little background "
            "on this topic. Mainly wants to pass the exam — give the answer and what is needed "
            "to score — but it must be spelled out simply, step by step, with examples."
        ),
    ),
    "scholar": Profile(
        id="scholar",
        split="train",
        rendered=(
            "A ninth-grader who reads dense material easily and has solid background on this "
            "topic. Wants to understand the underlying why and how, and the connections between "
            "ideas. Prefers a terse, high-level treatment without hand-holding or padding."
        ),
    ),
    "steady": Profile(
        id="steady",
        split="train",
        rendered=(
            "A capable ninth-grader with average background on this topic. Wants a correct "
            "answer with a brief justification, balanced toward exam needs. Does not need "
            "elaborate scaffolding, but does appreciate a one-line reason."
        ),
    ),
}


def render_profile(persona_id: str) -> str:
    return PERSONAS[persona_id].rendered


def train_personas() -> list:
    return [p for p in PERSONAS.values() if p.split == "train"]

In [ ]:
# ── Encoding (from src/rl/ropg_kd.py) ───────────────────────────────────────
import torch
import torch.nn.functional as F
from sentence_transformers.util import batch_to_device


def encode_texts(st_model, texts, device, instruction=None):
    """Encode *texts* through the same preprocess -> forward path as ``SentenceTransformer.encode``,
    but with gradients enabled so it can be used inside a training loop.

    *instruction*, if given, is rendered as the Qwen3-Embedding instruct prefix
    (``Instruct: {instruction}\nQuery: ``) and passed as the ``prompt`` kwarg so the
    input module can correctly mask the prompt tokens out of pooling (``prompt_length``).
    """
    prompt = f"Instruct: {instruction}\nQuery: " if instruction else None
    features = st_model.preprocess(texts, prompt=prompt)
    features = batch_to_device(features, device)
    out = st_model.forward(features)
    # Cast to fp32 so cosine similarities and the KL loss are computed at full
    # precision even when the forward ran under fp16 autocast (fp16 resolution
    # near 1.0 is too coarse for stable ranking).
    return F.normalize(out["sentence_embedding"], p=2, dim=-1).float()


def encode_docs_chunked(st_model, texts, device, micro_batch):
    """Encode *texts* in micro-batches of *micro_batch* and concatenate.

    Peak activation memory scales with the per-forward batch, so chunking the ~20-doc
    encode keeps one KD step within a 16 GB T4. The concatenated embeddings still sit
    in a single autograd graph, so the listwise loss is unchanged.
    """
    vecs = [
        encode_texts(st_model, texts[i : i + micro_batch], device)
        for i in range(0, len(texts), micro_batch)
    ]
    return torch.cat(vecs, dim=0)

## Helper functions

In [ ]:
import json
import logging
import math
import random
import shutil

import numpy as np

logging.basicConfig(
    force=True,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ── Data loading ─────────────────────────────────────────────────────────────
def load_groups(path):
    """Load (query, persona, docs) groups from a ROPG-KD JSONL file.

    Each line is ``{"query": ..., "persona_id": ..., "docs": [{"chunk_id", "text",
    "teacher_score"}, ...]}``. Groups with fewer than 2 docs are dropped (KL over a
    singleton distribution is degenerate).
    """
    groups = []
    n_dropped = 0
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        if len(rec.get("docs", [])) < 2:
            n_dropped += 1
            continue
        groups.append(rec)
    logger.info(
        "Loaded %d groups from %s (dropped %d with < 2 docs)", len(groups), path, n_dropped
    )
    return groups


def load_corpus(path):
    """Load the full chunk corpus (fields: chunk_id, text) used for validation retrieval."""
    corpus = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        corpus.append({"chunk_id": rec["chunk_id"], "text": rec["text"]})
    logger.info("Loaded %d corpus chunks from %s", len(corpus), path)
    return corpus


# ── KD loss ──────────────────────────────────────────────────────────────────
def _kd_loss(q_vec, d_vecs, scores, temperature):
    """Listwise KD loss: KL(teacher softmax || student softmax) over one group's docs."""
    sims = (q_vec @ d_vecs.T).squeeze(0)
    targets = F.softmax(scores / temperature, dim=0)
    log_probs = F.log_softmax(sims / temperature, dim=0)
    return F.kl_div(log_probs, targets, reduction="sum")


# ── Validation ───────────────────────────────────────────────────────────────
def evaluate(st_model, val_groups, corpus, device, kd_temperature, top_k, relevance_top_m, corpus_batch=32, amp_dtype=None):
    """Run validation: mean KD loss over val groups, plus Recall@K / MRR per persona and overall.

    Ranks the full corpus by cosine similarity to each val group's persona-conditioned query
    embedding; the relevant set for a group is its top-*relevance_top_m* docs by teacher_score
    (matched against the corpus by chunk_id).
    """
    from tqdm import tqdm

    st_model.eval()
    _amp_dtype = amp_dtype if amp_dtype is not None else torch.float16
    with torch.no_grad(), torch.autocast("cuda", enabled=device == "cuda", dtype=_amp_dtype):
        # Re-embed the full corpus with the current model. Group docs *are* corpus
        # chunks, so their embeddings are looked up from this matrix by chunk_id
        # instead of being re-encoded per group.
        corpus_texts = [c["text"] for c in corpus]
        corpus_ids = [c["chunk_id"] for c in corpus]
        corpus_vecs = []
        for i in tqdm(range(0, len(corpus_texts), corpus_batch), desc="  eval: corpus", leave=False):
            corpus_vecs.append(encode_texts(st_model, corpus_texts[i : i + corpus_batch], device))
        corpus_matrix = torch.cat(corpus_vecs, dim=0)  # (N, dim)
        chunk_id_to_row = {cid: i for i, cid in enumerate(corpus_ids)}

        # Batch-encode val queries per persona: same instruction means the same
        # prompt, so they can share a forward.
        by_persona = {}
        for gi, group in enumerate(val_groups):
            by_persona.setdefault(group["persona_id"], []).append(gi)
        query_vecs = {}
        for persona_id, idxs in by_persona.items():
            instruction = render_profile(persona_id)
            for j in range(0, len(idxs), corpus_batch):
                chunk = idxs[j : j + corpus_batch]
                vecs = encode_texts(
                    st_model,
                    [val_groups[gi]["query"] for gi in chunk],
                    device,
                    instruction=instruction,
                )
                for gi, vec in zip(chunk, vecs):
                    query_vecs[gi] = vec

        # Val KD loss, computed from the corpus embeddings.
        losses = []
        for gi, group in enumerate(val_groups):
            rows = []
            kept_scores = []
            for d in group["docs"]:
                row = chunk_id_to_row.get(d["chunk_id"])
                if row is not None:
                    rows.append(row)
                    kept_scores.append(d["teacher_score"])
            if len(rows) < 2:
                continue
            scores = torch.tensor(kept_scores, device=device, dtype=torch.float32)
            q_vec = query_vecs[gi].unsqueeze(0)
            losses.append(_kd_loss(q_vec, corpus_matrix[rows], scores, kd_temperature).item())
        val_loss = float(np.mean(losses)) if losses else float("nan")

        per_persona_recall = {}
        per_persona_mrr = {}
        all_recall = []
        all_mrr = []

        for gi, group in enumerate(val_groups):
            persona_id = group["persona_id"]
            relevant_docs = sorted(group["docs"], key=lambda d: d["teacher_score"], reverse=True)[
                :relevance_top_m
            ]
            relevant_rows = {
                chunk_id_to_row[d["chunk_id"]]
                for d in relevant_docs
                if d["chunk_id"] in chunk_id_to_row
            }
            if not relevant_rows:
                continue

            q_vec = query_vecs[gi].unsqueeze(0)
            sims = (q_vec @ corpus_matrix.T).squeeze(0)  # (N,)
            ranked = torch.argsort(sims, descending=True).tolist()

            top_k_rows = set(ranked[:top_k])
            recall = len(relevant_rows & top_k_rows) / len(relevant_rows)

            rr = 0.0
            for rank, row in enumerate(ranked, start=1):
                if row in relevant_rows:
                    rr = 1.0 / rank
                    break

            per_persona_recall.setdefault(persona_id, []).append(recall)
            per_persona_mrr.setdefault(persona_id, []).append(rr)
            all_recall.append(recall)
            all_mrr.append(rr)

    metrics = {
        "val_kd_loss": val_loss,
        "recall_at_k": {"overall": float(np.mean(all_recall)) if all_recall else 0.0},
        "mrr": {"overall": float(np.mean(all_mrr)) if all_mrr else 0.0},
    }
    for persona_id in per_persona_recall:
        metrics["recall_at_k"][persona_id] = float(np.mean(per_persona_recall[persona_id]))
        metrics["mrr"][persona_id] = float(np.mean(per_persona_mrr[persona_id]))

    logger.info(
        "Validation | KD loss: %.4f | Recall@K overall: %.4f | MRR overall: %.4f",
        val_loss,
        metrics["recall_at_k"]["overall"],
        metrics["mrr"]["overall"],
    )
    for persona_id in per_persona_recall:
        logger.info(
            "  persona=%s | Recall@K: %.4f | MRR: %.4f",
            persona_id,
            metrics["recall_at_k"][persona_id],
            metrics["mrr"][persona_id],
        )
    return metrics


def _is_better(candidate, current_best):
    """Best checkpoint = highest overall Recall@K; ties -> lower val KL; ties -> higher overall MRR."""
    if current_best is None:
        return True
    c_recall = candidate["recall_at_k"]["overall"]
    b_recall = current_best["recall_at_k"]["overall"]
    if c_recall != b_recall:
        return c_recall > b_recall
    if candidate["val_kd_loss"] != current_best["val_kd_loss"]:
        return candidate["val_kd_loss"] < current_best["val_kd_loss"]
    return candidate["mrr"]["overall"] > current_best["mrr"]["overall"]

## Run pipeline

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from sentence_transformers import SentenceTransformer
from transformers import get_linear_schedule_with_warmup

seed = CFG["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = CFG["training"]["device"]
precision = CFG["training"]["precision"]
# use_amp fires for both fp16 and bf16; GradScaler only needed for fp16
use_amp = device == "cuda" and precision in ("fp16", "bf16")
# bf16 Tensor Cores need Ampere+ (compute capability >= 8); fall back to fp16 on T4/V100
if device == "cuda" and torch.cuda.is_available():
    model_dtype = (
        torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    )
else:
    model_dtype = torch.float32
amp_dtype = torch.bfloat16 if precision == "bf16" else torch.float16
logger.info("Device: %s | precision: %s | model_dtype: %s | amp: %s", device, precision, model_dtype, use_amp)

train_groups = load_groups(CFG["data"]["train_path"])
val_groups = load_groups(CFG["data"]["val_path"])
corpus = load_corpus(CFG["data"]["corpus_path"])

# Load the encoder and wrap with a LoRA adapter. Base model is loaded in half
# precision to cut weight memory ~50%; LoRA adapter weights are cast back to fp32
# below so the optimizer and gradient math stay stable.
model_name = CFG["embedder"]["model"]
attn_impl = CFG["embedder"].get("attn_implementation", "sdpa")
st_model = SentenceTransformer(
    model_name, device=device, trust_remote_code=True,
    model_kwargs={"attn_implementation": attn_impl, "torch_dtype": model_dtype},
)
lora_cfg_dict = CFG["lora"]
# get_peft_model injects the LoRA layers and freezes the base weights *in place* on
# the passed model, so SentenceTransformer's forward path trains through them without
# reassignment (Transformer.auto_model is a read-only property in ST 5.6; assigning to
# it is silently shadowed by nn.Module.__setattr__). Keep the wrapper only for
# print_trainable_parameters() and adapter-only save_pretrained().
peft_model = get_peft_model(
    st_model[0].auto_model,
    LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=lora_cfg_dict["r"],
        lora_alpha=lora_cfg_dict["alpha"],
        lora_dropout=lora_cfg_dict["dropout"],
        target_modules=lora_cfg_dict["target_modules"],
    ),
)
peft_model.print_trainable_parameters()

# Base model is in half precision (frozen). LoRA adapter weights must stay fp32
# so the optimizer and gradient accumulation are numerically stable.
for param in peft_model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# One KD step keeps the activation graphs of a query + ~20 docs alive until the
# backward pass, which OOMs a 16 GB T4 without checkpointing. use_reentrant=False
# lets gradients flow even though only the LoRA params require grad.
if CFG["training"]["gradient_checkpointing"]:
    st_model[0].auto_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
# Cap tokenization length: the tokenizer default (32k) lets one long outlier
# blow up the whole padded batch; real chunks are far shorter than 8k tokens.
st_model[0].max_seq_length = CFG["embedder"]["max_seq_length"]

# Diagnostics: confirm checkpointing actually engaged and how big batches really
# are in tokens — the evidence needed to aim any further OOM fix.
auto_model = st_model[0].auto_model
logger.info(
    "Gradient checkpointing active: %s | attn implementation: %s",
    getattr(auto_model, "is_gradient_checkpointing", "unknown"),
    getattr(auto_model.config, "_attn_implementation", "unknown"),
)
tokenizer = st_model[0].tokenizer
if tokenizer is not None:
    lengths = sorted(len(tokenizer(c["text"])["input_ids"]) for c in corpus)
    logger.info(
        "Corpus token lengths | max: %d | p95: %d | median: %d",
        lengths[-1],
        lengths[int(0.95 * (len(lengths) - 1))],
        lengths[len(lengths) // 2],
    )

trainable_params = [p for p in st_model.parameters() if p.requires_grad]
train_cfg = CFG["training"]
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=train_cfg["lr"],
    weight_decay=train_cfg["weight_decay"],
)

n_epochs = train_cfg["epochs"]
batch_size = train_cfg["batch_size"]
kd_temp = train_cfg["kd_temperature"]
grad_clip = train_cfg["grad_clip"]
warmup_ratio = train_cfg["warmup_ratio"]

n_steps_per_epoch = max(1, math.ceil(len(train_groups) / batch_size))
total_steps = n_steps_per_epoch * n_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * warmup_ratio),
    num_training_steps=total_steps,
)
# GradScaler only for fp16; bf16 has sufficient dynamic range without it
scaler = torch.amp.GradScaler("cuda") if (use_amp and precision == "fp16") else None

top_k = CFG["eval"]["top_k"]
relevance_top_m = CFG["eval"]["relevance_top_m"]
doc_micro_batch = train_cfg["doc_micro_batch"]

# DataLoader batches batch_size groups per step; workers prefetch while GPU computes.
import sys
from torch.utils.data import DataLoader


class KDGroupDataset:
    def __init__(self, groups):
        self.groups = groups

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        return self.groups[idx]


_num_workers = 2 if sys.platform != "darwin" else 0
train_loader = DataLoader(
    KDGroupDataset(train_groups),
    batch_size=batch_size,
    shuffle=True,
    num_workers=_num_workers,
    collate_fn=list,
    prefetch_factor=2 if _num_workers > 0 else None,
    persistent_workers=_num_workers > 0,
)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()


In [ ]:
# Health check: encode one train group, compute the KD loss and assert it is finite, then
# run the epoch-0 baseline validation. This catches NaN/shape/path problems before
# committing to the full training run.
import contextlib

sample_group = train_groups[0]
sample_scores = torch.tensor(
    [d["teacher_score"] for d in sample_group["docs"]], device=device, dtype=torch.float32
)
sample_instruction = render_profile(sample_group["persona_id"])

st_model.train()
sample_doc_texts = [d["text"] for d in sample_group["docs"]]
ctx = torch.amp.autocast("cuda", dtype=amp_dtype) if use_amp else contextlib.nullcontext()
with ctx:
    q_vec = encode_texts(st_model, [sample_group["query"]], device, instruction=sample_instruction)
    d_vecs = encode_docs_chunked(st_model, sample_doc_texts, device, doc_micro_batch)
    health_loss = _kd_loss(q_vec, d_vecs, sample_scores, kd_temp)

assert torch.isfinite(health_loss), f"Health check failed: KD loss is not finite ({health_loss})"
print(f"Health check KD loss: {health_loss.item():.4f}")

# Exercise a full backward once so the true training peak shows up here, in 30
# seconds, instead of minutes into epoch 1.
health_loss.backward()
optimizer.zero_grad(set_to_none=True)
if device == "cuda":
    print(
        f"Health-check peak memory | allocated: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB"
        f" | reserved: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB"
    )
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

logger.info("Running epoch-0 baseline validation (frozen encoder)...")
baseline_metrics = evaluate(st_model, val_groups, corpus, device, kd_temp, top_k, relevance_top_m, doc_micro_batch, amp_dtype=amp_dtype)
print(json.dumps(baseline_metrics, indent=2, ensure_ascii=False))


In [ ]:
import contextlib

from tqdm.auto import tqdm

epoch_metrics = []
best_metrics = None
best_epoch = 0

checkpoint_dir = Path(CFG["checkpoint_dir"])

for epoch in range(1, n_epochs + 1):
    st_model.train()
    epoch_loss = 0.0
    n_groups_seen = 0

    for mini_batch in tqdm(train_loader, desc=f"Epoch {epoch}/{n_epochs}"):
        # Collect all doc texts and scores up front so they can be encoded in one
        # chunked call instead of one call per group (reduces Python overhead and
        # gives encode_docs_chunked larger, GPU-friendly batches).
        by_persona = {}
        all_doc_texts = []
        doc_offsets = []
        all_scores_list = []

        for gi, group in enumerate(mini_batch):
            by_persona.setdefault(group["persona_id"], []).append((gi, group["query"]))
            docs = sorted(group["docs"], key=lambda d: d["teacher_score"], reverse=True)[:8]
            doc_offsets.append((len(all_doc_texts), len(docs)))
            all_doc_texts.extend(d["text"] for d in docs)
            all_scores_list.append(
                torch.tensor(
                    [d["teacher_score"] for d in docs], device=device, dtype=torch.float32
                )
            )

        ctx = (
            torch.amp.autocast("cuda", dtype=amp_dtype)
            if use_amp
            else contextlib.nullcontext()
        )
        with ctx:
            # Encode queries in sub-batches grouped by persona so all queries in
            # a sub-batch share the same instruction prefix and prompt_length.
            q_vecs = [None] * len(mini_batch)
            for persona_id, idx_queries in by_persona.items():
                idxs, queries = zip(*idx_queries)
                vecs = encode_texts(
                    st_model, list(queries), device,
                    instruction=render_profile(persona_id),
                )
                for gi, vec in zip(idxs, vecs):
                    q_vecs[gi] = vec

            # Encode all docs from the batch in one chunked call.
            all_d_vecs = encode_docs_chunked(st_model, all_doc_texts, device, doc_micro_batch)

            # Sum per-group KD losses; single backward per optimizer step replaces
            # the old gradient-accumulation pattern (was 1 backward per group).
            step_loss = sum(
                _kd_loss(
                    q_vecs[gi].unsqueeze(0),
                    all_d_vecs[offset : offset + n],
                    all_scores_list[gi],
                    kd_temp,
                )
                for gi, (offset, n) in enumerate(doc_offsets)
            ) / len(mini_batch)

        if use_amp:
            scaler.scale(step_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            step_loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, grad_clip)
            optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        epoch_loss += step_loss.item() * len(mini_batch)
        n_groups_seen += len(mini_batch)

    avg_train_loss = epoch_loss / max(n_groups_seen, 1)
    logger.info("Epoch %d/%d | avg train KL loss: %.4f", epoch, n_epochs, avg_train_loss)

    if device == "cuda":
        logger.info(
            "CUDA peak memory | allocated: %.2f GB | reserved: %.2f GB",
            torch.cuda.max_memory_allocated() / 1e9,
            torch.cuda.max_memory_reserved() / 1e9,
        )
        torch.cuda.empty_cache()
    val_metrics = evaluate(st_model, val_groups, corpus, device, kd_temp, top_k, relevance_top_m, doc_micro_batch, amp_dtype=amp_dtype)
    val_metrics["epoch"] = epoch
    val_metrics["train_kd_loss"] = avg_train_loss
    epoch_metrics.append(val_metrics)

    epoch_dir = checkpoint_dir / f"epoch{epoch}"
    peft_model.save_pretrained(str(epoch_dir))
    (epoch_dir / "metrics.json").write_text(
        json.dumps(val_metrics, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    logger.info("Checkpoint saved: %s", epoch_dir)

    if _is_better(val_metrics, best_metrics):
        best_metrics = val_metrics
        best_epoch = epoch

best_dir = checkpoint_dir / "best"
if best_epoch > 0:
    source_dir = checkpoint_dir / f"epoch{best_epoch}"
    if best_dir.exists():
        shutil.rmtree(best_dir)
    shutil.copytree(source_dir, best_dir)
    logger.info("Best checkpoint (epoch %d) copied to %s", best_epoch, best_dir)

training_log = {
    "config": CFG,
    "seed": seed,
    "baseline_metrics": baseline_metrics,
    "epoch_metrics": epoch_metrics,
    "best_epoch": best_epoch,
    "best_epoch_reason": (
        "highest overall Recall@K; ties broken by lower val KL loss, then higher overall MRR"
    ),
}
(checkpoint_dir / "training_log.json").write_text(
    json.dumps(training_log, ensure_ascii=False, indent=2), encoding="utf-8"
)
logger.info("Training log written: %s", checkpoint_dir / "training_log.json")


## Download

In [ ]:
import subprocess
from pathlib import Path

if RUNTIME != "colab":
    subprocess.run(
        [
            "zip", "-r", "ropg_kd_adapter.zip",
            "ropg_kd_checkpoints/best",
            "ropg_kd_checkpoints/training_log.json",
        ],
        cwd=str(Path(OUTPUT_DIR).parent),
        check=True,
    )
    print("ropg_kd_adapter.zip created.")
else:
    print("Colab: checkpoints already on Google Drive — skipping zip.")